# Advanced RAG: Maximizing Accuracy for Medical Domain

Basic RAG: embed → search → answer. Works for general Q&A.

But in medicine, "close enough" can kill. This notebook covers techniques to make RAG **precise and reliable** — using a medical knowledge base as the running example.

In [ ]:
import numpy as np
np.random.seed(42)

# ═══════════════════════════════════════════════════════════════
# Our Medical Knowledge Base
# ═══════════════════════════════════════════════════════════════

medical_docs = [
    {
        "id": "D001",
        "title": "Metformin Prescribing Guidelines",
        "category": "medication",
        "drug": "metformin",
        "updated": "2026-03-15",
        "text": "Metformin is first-line therapy for type 2 diabetes. Starting dose: 500mg once daily with meals. Maximum dose: 2550mg/day in divided doses. Contraindicated in patients with eGFR < 30 mL/min. Hold 48 hours before and after contrast dye procedures. Common side effects: nausea, diarrhea, abdominal pain. Rare but serious: lactic acidosis (risk increases with renal impairment, hepatic disease, or alcohol use)."
    },
    {
        "id": "D002",
        "title": "Metformin Drug Interactions",
        "category": "medication",
        "drug": "metformin",
        "updated": "2026-03-15",
        "text": "Metformin interactions: Alcohol increases lactic acidosis risk. Iodinated contrast agents: hold metformin 48hr before, resume 48hr after if renal function stable. Carbonic anhydrase inhibitors (topiramate, zonisamide): increase lactic acidosis risk. Cimetidine: increases metformin levels by 60%. ACE inhibitors: may enhance hypoglycemic effect. Monitor renal function with concurrent nephrotoxic drugs."
    },
    {
        "id": "D003",
        "title": "Insulin Dosing for Type 2 Diabetes",
        "category": "medication",
        "drug": "insulin",
        "updated": "2026-02-20",
        "text": "Basal insulin initiation: Start 10 units/day or 0.1-0.2 units/kg/day. Titrate by 2 units every 3 days until fasting glucose 80-130 mg/dL. If HbA1c remains above target after 3-6 months on basal insulin, add prandial insulin. Hypoglycemia management: blood glucose < 70 mg/dL, treat with 15g fast-acting carbohydrate."
    },
    {
        "id": "D004",
        "title": "Warfarin Management Protocol",
        "category": "medication",
        "drug": "warfarin",
        "updated": "2026-01-10",
        "text": "Warfarin target INR: 2.0-3.0 for most indications. For mechanical heart valves: target INR 2.5-3.5. Check INR every 1-4 weeks when stable. Bridging with heparin for invasive procedures. Major food interactions: vitamin K-rich foods (leafy greens) reduce effectiveness. Numerous drug interactions: NSAIDs increase bleeding risk, antibiotics may increase INR."
    },
    {
        "id": "D005",
        "title": "Acute MI Management Protocol",
        "category": "emergency",
        "drug": None,
        "updated": "2026-04-01",
        "text": "Acute myocardial infarction (MI): STEMI protocol: Door-to-balloon time < 90 minutes. Immediate: Aspirin 325mg chewed, Heparin bolus, Nitroglycerin if SBP > 90. Clopidogrel 600mg loading dose. Morphine for pain unresponsive to nitro. NSTEMI: risk stratify with TIMI score, anticoagulation, early invasive strategy for high-risk patients."
    },
    {
        "id": "D006",
        "title": "Metformin Use in Renal Impairment",
        "category": "medication",
        "drug": "metformin",
        "updated": "2026-04-10",
        "text": "Metformin renal dosing: eGFR >= 45: no dose adjustment needed. eGFR 30-44: reduce maximum dose to 1000mg/day, monitor renal function every 3 months. eGFR < 30: CONTRAINDICATED — do not initiate or continue. For patients on metformin whose eGFR falls below 30: discontinue immediately. Risk of lactic acidosis increases exponentially below eGFR 30."
    },
    {
        "id": "D007",
        "title": "Penicillin Allergy Cross-Reactivity",
        "category": "allergy",
        "drug": "penicillin",
        "updated": "2026-03-01",
        "text": "Penicillin allergy: true IgE-mediated allergy in < 5% of reported cases. Cross-reactivity with cephalosporins: 1-2% (previously overestimated at 10%). Safe alternatives: azithromycin, fluoroquinolones. For confirmed penicillin allergy needing cephalosporins: skin testing recommended. Carbapenems: < 1% cross-reactivity, generally safe."
    },
    {
        "id": "D008",
        "title": "Diabetes Monitoring Schedule",
        "category": "monitoring",
        "drug": None,
        "updated": "2026-02-15",
        "text": "Type 2 diabetes monitoring: HbA1c every 3 months until stable, then every 6 months. Fasting lipid panel annually. Serum creatinine and eGFR annually (more frequent if on metformin or with CKD). Urine albumin-to-creatinine ratio annually. Dilated eye exam annually. Foot exam every visit. Blood pressure target: < 130/80 mmHg."
    },
]

print(f"Medical Knowledge Base: {len(medical_docs)} documents\n")
for doc in medical_docs:
    print(f"  [{doc['id']}] {doc['title']}")
    print(f"        category={doc['category']}, drug={doc['drug']}, updated={doc['updated']}")

---
## Problem 1: Basic RAG Fails on Precise Queries

Let's see why basic vector search isn't good enough for medicine.

In [ ]:
# Simulate embeddings (in reality: call embedding API)
# We'll design them to show realistic failure modes

embed_dim = 32

def make_medical_embedding(doc):
    """Simulate a medical embedding model."""
    np.random.seed(hash(doc['id']) % 2**31)
    vec = np.random.randn(embed_dim) * 0.3
    
    # Simulate semantic clustering
    if doc['drug'] == 'metformin':
        vec[0:4] += 1.5
    if doc['drug'] == 'insulin':
        vec[0:4] += 1.0  # similar to metformin (both diabetes)
        vec[4:8] += 1.0
    if doc['drug'] == 'warfarin':
        vec[8:12] += 1.5
    if doc['category'] == 'emergency':
        vec[12:16] += 1.5
    if 'renal' in doc['text'].lower() or 'eGFR' in doc['text']:
        vec[16:20] += 1.2
    if 'interaction' in doc['text'].lower() or 'drug' in doc['title'].lower():
        vec[20:24] += 1.0
    if 'dose' in doc['text'].lower() or 'dosing' in doc['title'].lower():
        vec[24:28] += 1.0
    
    return vec / np.linalg.norm(vec)

def make_query_embedding(query):
    """Simulate query embedding."""
    np.random.seed(hash(query) % 2**31)
    vec = np.random.randn(embed_dim) * 0.3
    
    if 'metformin' in query.lower():
        vec[0:4] += 1.5
    if 'insulin' in query.lower():
        vec[0:4] += 1.0
        vec[4:8] += 1.0
    if 'renal' in query.lower() or 'kidney' in query.lower() or 'egfr' in query.lower():
        vec[16:20] += 1.2
    if 'interaction' in query.lower() or 'drug' in query.lower():
        vec[20:24] += 1.0
    if 'dose' in query.lower() or 'dosing' in query.lower() or 'maximum' in query.lower():
        vec[24:28] += 1.0
    if 'contrast' in query.lower():
        vec[20:24] += 0.8
    if 'emergency' in query.lower() or 'acute' in query.lower() or 'MI' in query:
        vec[12:16] += 1.5
    
    return vec / np.linalg.norm(vec)

# Index all documents
doc_embeddings = [(doc, make_medical_embedding(doc)) for doc in medical_docs]

def basic_search(query, top_k=3):
    """Basic vector search — just cosine similarity."""
    q_vec = make_query_embedding(query)
    scores = []
    for doc, vec in doc_embeddings:
        sim = np.dot(q_vec, vec)
        scores.append((doc, sim))
    scores.sort(key=lambda x: -x[1])
    return scores[:top_k]

# Test: precise medical query
query = "What is the maximum metformin dose for a patient with eGFR 35?"
print(f"Query: '{query}'\n")
print(f"Basic vector search results:\n")

results = basic_search(query)
for i, (doc, score) in enumerate(results, 1):
    print(f"  {i}. [{score:.3f}] {doc['title']}")
    print(f"     (category={doc['category']}, drug={doc['drug']})")

print(f"\n  Problem: The query needs SPECIFICALLY the renal dosing doc (D006).")
print(f"  But basic search might return general metformin docs too,")
print(f"  which say 'max 2550mg/day' — WRONG for this patient!")
print(f"\n  In medicine, returning the general doc could be DANGEROUS.")

---
## Technique 1: Metadata Filtering (Pre-filter Before Vector Search)

Don't just search by meaning — use **structured filters** to narrow down first.

```
Basic RAG:    search ALL docs by similarity
Filtered RAG: filter by metadata FIRST, then search by similarity within results
```

This is like SQL WHERE + vector search combined.

In [ ]:
# Metadata filtering
print("Technique 1: Metadata Filtering\n")
print("═" * 60)

def filtered_search(query, filters=None, top_k=3):
    """
    Search with metadata filters applied BEFORE vector similarity.
    filters: dict of {field: value} that must match.
    """
    q_vec = make_query_embedding(query)
    
    scores = []
    for doc, vec in doc_embeddings:
        # Apply metadata filters first
        if filters:
            skip = False
            for field, value in filters.items():
                if field == 'drug' and doc.get('drug') != value:
                    skip = True
                if field == 'category' and doc.get('category') != value:
                    skip = True
                if field == 'updated_after':
                    if doc.get('updated', '') < value:
                        skip = True
            if skip:
                continue
        
        sim = np.dot(q_vec, vec)
        scores.append((doc, sim))
    
    scores.sort(key=lambda x: -x[1])
    return scores[:top_k]

# Same query, but now with metadata filter
query = "What is the maximum metformin dose for a patient with eGFR 35?"
filters = {"drug": "metformin"}

print(f"\n  Query: '{query}'")
print(f"  Filter: drug='metformin'\n")

results = filtered_search(query, filters=filters)
print(f"  Results (only metformin docs, ranked by relevance):\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"    {i}. [{score:.3f}] {doc['title']}")
    relevant = '← ANSWER HERE' if 'Renal' in doc['title'] else ''
    print(f"       {doc['text'][:80]}... {relevant}")

print(f"\n  By filtering to metformin-only docs, we eliminated noise")
print(f"  (warfarin, insulin, emergency protocols, etc.)")
print(f"\n  In production: vector DBs support this natively:")
print(f"    pinecone.query(vector=q_vec, filter={{'drug': 'metformin'}})")
print(f"    chromadb.query(where={{'drug': 'metformin'}})")

---
## Technique 2: Hybrid Search (Vector + Keyword)

Vector search finds **meaning**. But sometimes you need **exact terms**.

```
"eGFR < 30 contraindication" — the exact number matters!
Vector search might match "renal impairment" (close meaning)
but miss the specific eGFR threshold.
```

Hybrid = combine both, get the best of each.

In [ ]:
# Hybrid search: vector + keyword (BM25)
print("Technique 2: Hybrid Search (Vector + Keyword)\n")
print("═" * 60)

def keyword_score(query, doc_text):
    """Simple keyword matching score (BM25-like)."""
    query_terms = query.lower().split()
    doc_lower = doc_text.lower()
    
    matches = 0
    for term in query_terms:
        if term in doc_lower:
            matches += 1
        # Boost for exact medical terms
        if term in ['egfr', 'metformin', 'contraindicated', '30', '2550']:
            if term in doc_lower:
                matches += 2  # extra weight for precise terms
    
    return matches / len(query_terms)

def hybrid_search(query, filters=None, top_k=3, vector_weight=0.5, keyword_weight=0.5):
    """
    Combine vector similarity + keyword matching.
    """
    q_vec = make_query_embedding(query)
    
    scores = []
    for doc, vec in doc_embeddings:
        # Apply filters
        if filters:
            skip = False
            for field, value in filters.items():
                if field == 'drug' and doc.get('drug') != value:
                    skip = True
            if skip:
                continue
        
        # Vector score
        v_score = np.dot(q_vec, vec)
        
        # Keyword score
        k_score = keyword_score(query, doc['text'])
        
        # Combined score
        combined = vector_weight * v_score + keyword_weight * k_score
        scores.append((doc, combined, v_score, k_score))
    
    scores.sort(key=lambda x: -x[1])
    return scores[:top_k]

# Test with a query where exact terms matter
query = "metformin eGFR 30 contraindicated"
print(f"\n  Query: '{query}'\n")

print(f"  Vector-only search:")
v_results = basic_search(query)
for i, (doc, score) in enumerate(v_results[:3], 1):
    print(f"    {i}. [{score:.3f}] {doc['title']}")

print(f"\n  Hybrid search (vector + keyword):")
h_results = hybrid_search(query, filters={"drug": "metformin"})
for i, (doc, combined, v_score, k_score) in enumerate(h_results[:3], 1):
    print(f"    {i}. [combined={combined:.3f}] {doc['title']}")
    print(f"       vector={v_score:.3f}, keyword={k_score:.3f}")

print(f"\n  Keyword matching catches exact terms like 'eGFR', '30', 'contraindicated'")
print(f"  that vector search might dilute across semantically similar docs.")
print(f"\n  For medical queries: keyword_weight should be HIGHER (0.6-0.7)")
print(f"  because exact drug names, dosages, and lab values matter more than 'vibe'.")

---
## Technique 3: Re-Ranking (Two-Stage Retrieval)

Stage 1: Fast search retrieves top-20 candidates (cheap, rough)
Stage 2: Expensive model re-scores those 20 for precise ranking

```
Why? The embedding model compresses a whole sentence into ONE vector.
A cross-encoder compares query and document TOGETHER — much more accurate.
```

In [ ]:
# Re-ranking simulation
print("Technique 3: Re-Ranking with Cross-Encoder\n")
print("═" * 60)

def cross_encoder_score(query, doc_text):
    """
    Simulates a cross-encoder re-ranker.
    
    Real cross-encoder: feeds [query + document] through a transformer TOGETHER,
    so it can compare them word-by-word (much more accurate than separate embeddings).
    
    Here we simulate with detailed keyword/context matching.
    """
    score = 0.0
    query_lower = query.lower()
    doc_lower = doc_text.lower()
    
    # Exact phrase matching (cross-encoder is great at this)
    if 'egfr' in query_lower and 'egfr' in doc_lower:
        score += 3.0
    if '30' in query_lower and '30' in doc_lower:
        score += 2.0
    if 'contraindicated' in query_lower and 'contraindicated' in doc_lower:
        score += 3.0
    if 'maximum' in query_lower and ('maximum' in doc_lower or 'max' in doc_lower):
        score += 2.0
    if 'dose' in query_lower and ('dose' in doc_lower or 'dosing' in doc_lower):
        score += 1.5
    
    # Context matching (understands the COMBINATION of terms)
    if 'egfr' in query_lower and 'renal' in doc_lower:
        score += 2.0  # understands eGFR relates to renal
    if 'maximum' in query_lower and 'dose' in query_lower:
        if '1000mg' in doc_lower or '2550mg' in doc_lower:
            score += 3.0  # found an actual dosage number
    
    # Penalize irrelevant docs that share some keywords
    if 'metformin' in query_lower and 'metformin' not in doc_lower:
        score -= 5.0
    
    return score

def search_with_reranking(query, filters=None, initial_k=5, final_k=2):
    """
    Stage 1: Fast vector search (top initial_k)
    Stage 2: Re-rank with cross-encoder (return top final_k)
    """
    # Stage 1: cheap vector search
    candidates = filtered_search(query, filters=filters, top_k=initial_k)
    
    # Stage 2: expensive re-ranking
    reranked = []
    for doc, vec_score in candidates:
        rerank_score = cross_encoder_score(query, doc['text'])
        reranked.append((doc, rerank_score, vec_score))
    
    reranked.sort(key=lambda x: -x[1])
    return reranked[:final_k]

# Test
query = "What is the maximum metformin dose for eGFR 35?"
print(f"\n  Query: '{query}'\n")

print(f"  Stage 1 — Vector search (fast, rough):")
candidates = filtered_search(query, filters={"drug": "metformin"}, top_k=4)
for i, (doc, score) in enumerate(candidates, 1):
    print(f"    {i}. [{score:.3f}] {doc['title']}")

print(f"\n  Stage 2 — Re-rank (slow, precise):")
final = search_with_reranking(query, filters={"drug": "metformin"}, initial_k=4, final_k=2)
for i, (doc, rerank_score, vec_score) in enumerate(final, 1):
    print(f"    {i}. [rerank={rerank_score:.1f}] {doc['title']}")
    print(f"       '{doc['text'][:100]}...'")

print(f"\n  The renal dosing doc scores highest because it contains")
print(f"  EXACTLY what was asked: eGFR + 30 + dose + contraindicated.")
print(f"\n  Cross-encoder models: ms-marco-MiniLM, BGE-reranker, Cohere rerank")

---
## Technique 4: Query Decomposition

Complex medical queries often combine multiple sub-questions:

```
"Patient on metformin with eGFR 35 needs CT with contrast. What to do?"

This requires:
  1. Metformin + renal dosing (eGFR 35 = reduce dose)
  2. Metformin + contrast interaction (hold 48 hours)
  3. Combining both answers
```

One vector search won't find both pieces. Decompose first.

In [ ]:
# Query decomposition
print("Technique 4: Query Decomposition\n")
print("═" * 60)

complex_query = "Patient on metformin with eGFR 35 needs CT with contrast. What to do?"
print(f"\n  Complex query: '{complex_query}'\n")

# Step 1: LLM decomposes the query (simulated)
sub_queries = [
    "metformin dosing for eGFR 35 renal impairment",
    "metformin contrast dye interaction hold",
]

print(f"  Step 1: LLM decomposes into sub-queries:")
for i, sq in enumerate(sub_queries, 1):
    print(f"    Sub-query {i}: '{sq}'")

# Step 2: Search for each sub-query
print(f"\n  Step 2: Search for each sub-query separately:\n")
all_retrieved = []

for i, sq in enumerate(sub_queries, 1):
    results = hybrid_search(sq, filters={"drug": "metformin"}, top_k=2)
    print(f"    Sub-query {i}: '{sq}'")
    for doc, combined, v_score, k_score in results[:1]:  # top 1
        print(f"      → [{doc['id']}] {doc['title']}")
        all_retrieved.append(doc)
    print()

# Step 3: Deduplicate and build context
unique_docs = {doc['id']: doc for doc in all_retrieved}
print(f"  Step 3: Combine results ({len(unique_docs)} unique docs):\n")

for doc_id, doc in unique_docs.items():
    print(f"    [{doc_id}] {doc['title']}")
    print(f"    '{doc['text'][:100]}...'\n")

# Step 4: Build final prompt
print(f"  Step 4: LLM synthesizes answer from ALL retrieved docs:\n")
print(f"    'For a patient with eGFR 35 on metformin needing CT contrast:")
print(f"     1. Reduce metformin to max 1000mg/day (eGFR 30-44 range) [D006]")
print(f"     2. Hold metformin 48 hours before contrast procedure [D002]")
print(f"     3. Resume 48 hours after if renal function stable [D002]")
print(f"     4. Monitor renal function after procedure [D006]'")
print(f"\n  Without decomposition, a single search might only find ONE of these docs.")

---
## Technique 5: Source Verification and Citation

In medicine, EVERY claim must be traceable to a source. The LLM must:
1. Only state what's in the retrieved documents
2. Cite which document supports each claim
3. Say "I don't know" if the answer isn't in the context

In [ ]:
# Citation and verification
print("Technique 5: Forced Citation and Verification\n")
print("═" * 60)

# The prompt engineering that enforces citation
system_prompt = """You are a medical information assistant. STRICT RULES:

1. ONLY answer using information from the provided documents.
2. CITE the document ID [D001], [D002], etc. for EVERY factual claim.
3. If the documents don't contain the answer, say: "NOT FOUND IN DATABASE."
4. NEVER infer, guess, or use general knowledge.
5. If information seems outdated (check the 'updated' date), WARN the user.
6. For dosing: always include the EXACT numbers from the source document.

Format your response as:
- Answer: [your answer with citations]
- Sources: [list of document IDs used]
- Confidence: HIGH (exact match found) / MEDIUM (related info) / LOW (partial)
- Caveats: [any limitations or warnings]"""

print(f"  System prompt for medical RAG:\n")
for line in system_prompt.split('\n'):
    print(f"  │ {line}")

print(f"\n\n  Example output with citations:\n")
print(f"  ┌─────────────────────────────────────────────────────────────┐")
print(f"  │ Answer: For eGFR 30-44, reduce metformin maximum dose to   │")
print(f"  │ 1000mg/day [D006]. Monitor renal function every 3 months   │")
print(f"  │ [D006]. If eGFR falls below 30, discontinue immediately    │")
print(f"  │ [D006].                                                     │")
print(f"  │                                                             │")
print(f"  │ Sources: [D006] Metformin Use in Renal Impairment          │")
print(f"  │          (updated: 2026-04-10)                              │")
print(f"  │ Confidence: HIGH                                            │")
print(f"  │ Caveats: Consult nephrology if eGFR declining rapidly.     │")
print(f"  └─────────────────────────────────────────────────────────────┘")

print(f"\n  Example when answer NOT found:\n")
print(f"  ┌─────────────────────────────────────────────────────────────┐")
print(f"  │ Answer: NOT FOUND IN DATABASE.                              │")
print(f"  │ The knowledge base does not contain information about       │")
print(f"  │ metformin use during pregnancy.                             │")
print(f"  │                                                             │")
print(f"  │ Confidence: N/A                                             │")
print(f"  │ Caveats: Please consult a specialist or refer to current   │")
print(f"  │ obstetric guidelines.                                       │")
print(f"  └─────────────────────────────────────────────────────────────┘")

---
## Technique 6: Confidence Scoring and Fallback

Not all retrievals are equal. The system should know when it's unsure.

In [ ]:
# Confidence scoring
print("Technique 6: Confidence Scoring\n")
print("═" * 60)

def search_with_confidence(query, filters=None, threshold=0.5):
    """
    Search and assess confidence level.
    If best match is below threshold → flag as low confidence.
    """
    results = hybrid_search(query, filters=filters, top_k=3)
    
    if not results:
        return [], "NONE", "No documents matched filters"
    
    best_score = results[0][1]  # combined score of top result
    
    # Assess confidence
    if best_score > 0.8:
        confidence = "HIGH"
        action = "Answer directly with citation"
    elif best_score > 0.5:
        confidence = "MEDIUM"
        action = "Answer with caveat: 'Based on partial match...'"
    elif best_score > 0.3:
        confidence = "LOW"
        action = "Show results but recommend human review"
    else:
        confidence = "NONE"
        action = "Do NOT answer. Escalate to physician."
    
    return results, confidence, action

# Test with different queries
test_queries = [
    ("metformin maximum dose eGFR 35", {"drug": "metformin"}),
    ("metformin interaction with topiramate", {"drug": "metformin"}),
    ("can metformin be used during pregnancy", {"drug": "metformin"}),
]

print(f"\n  Query Confidence Assessment:\n")
for query, filters in test_queries:
    results, confidence, action = search_with_confidence(query, filters)
    
    top_title = results[0][0]['title'] if results else 'None'
    top_score = results[0][1] if results else 0
    
    print(f"  Query: '{query}'")
    print(f"    Best match: {top_title} (score={top_score:.3f})")
    print(f"    Confidence: {confidence}")
    print(f"    Action: {action}\n")

print(f"  Critical for medicine:")
print(f"    • Low confidence → DON'T GUESS. Say 'I don't know.'")
print(f"    • Route to human expert instead of making up an answer.")
print(f"    • Log low-confidence queries → identify knowledge base gaps.")

---
## Technique 7: Temporal Awareness (Document Freshness)

Medical guidelines get updated. Old information can be dangerous.

```
2020 guideline: "eGFR < 45: contraindicated"
2024 guideline: "eGFR < 30: contraindicated"  ← threshold changed!
```

The system must prefer the most recent document.

In [ ]:
# Temporal awareness
print("Technique 7: Temporal Awareness\n")
print("═" * 60)

def time_boosted_search(query, filters=None, top_k=3, recency_weight=0.2):
    """
    Boost recent documents in search results.
    """
    from datetime import datetime
    today = datetime(2026, 5, 5)
    
    results = hybrid_search(query, filters=filters, top_k=top_k * 2)  # get more candidates
    
    boosted = []
    for doc, combined, v_score, k_score in results:
        # Calculate recency boost (newer = higher)
        doc_date = datetime.strptime(doc['updated'], '%Y-%m-%d')
        days_old = (today - doc_date).days
        recency_score = max(0, 1.0 - days_old / 365)  # 0-1, decays over a year
        
        final_score = combined * (1 - recency_weight) + recency_score * recency_weight
        boosted.append((doc, final_score, combined, recency_score))
    
    boosted.sort(key=lambda x: -x[1])
    return boosted[:top_k]

query = "metformin renal dosing guidelines"
print(f"\n  Query: '{query}'\n")

results = time_boosted_search(query, filters={"drug": "metformin"})
print(f"  {'Doc':<5} {'Title':<35} {'Score':>6} {'Relevance':>10} {'Recency':>8} {'Updated'}")
print(f"  {'─'*5} {'─'*35} {'─'*6} {'─'*10} {'─'*8} {'─'*10}")

for doc, final, relevance, recency in results:
    print(f"  {doc['id']:<5} {doc['title']:<35} {final:>6.3f} {relevance:>10.3f} {recency:>8.3f} {doc['updated']}")

print(f"\n  Newer documents get a boost — critical when guidelines change.")
print(f"\n  Additional safeguard: WARN if document is older than 1 year:")
print(f"    '⚠ This guideline was last updated 14 months ago.")
print(f"     Please verify against current standards.'")

---
## Putting It All Together: Production Medical RAG

```
User query
    ↓
1. Query decomposition (LLM splits complex questions)
    ↓
2. Metadata filtering (drug, category, date range)
    ↓
3. Hybrid search (vector + keyword for each sub-query)
    ↓
4. Re-ranking (cross-encoder scores top candidates)
    ↓
5. Temporal boost (prefer recent guidelines)
    ↓
6. Confidence check (score > threshold?)
    ├── YES → Build prompt with citations, generate answer
    └── NO  → "Not found. Escalating to physician."
         ↓
7. Output with forced citations and caveats
```

In [ ]:
# Full pipeline
print("Full Medical RAG Pipeline — End to End\n")
print("═" * 60)

def medical_rag_pipeline(query):
    """Complete medical RAG with all safety features."""
    print(f"\n  Query: '{query}'\n")
    
    # 1. Extract metadata from query (in reality: LLM or NER model)
    detected_drug = None
    for drug in ['metformin', 'insulin', 'warfarin', 'penicillin']:
        if drug in query.lower():
            detected_drug = drug
    
    filters = {}
    if detected_drug:
        filters['drug'] = detected_drug
    
    print(f"  Step 1 — Extract entities: drug={detected_drug}")
    
    # 2. Decompose if complex (simplified: check for multiple topics)
    sub_queries = [query]  # default: use as-is
    if 'contrast' in query.lower() and 'egfr' in query.lower():
        sub_queries = [
            f"{detected_drug} renal dosing eGFR",
            f"{detected_drug} contrast dye interaction",
        ]
        print(f"  Step 2 — Decomposed into {len(sub_queries)} sub-queries")
    else:
        print(f"  Step 2 — Single query (no decomposition needed)")
    
    # 3. Search for each sub-query
    all_docs = {}
    for sq in sub_queries:
        results = time_boosted_search(sq, filters=filters, top_k=2)
        for doc, score, relevance, recency in results:
            if doc['id'] not in all_docs or score > all_docs[doc['id']][1]:
                all_docs[doc['id']] = (doc, score)
    
    print(f"  Step 3 — Retrieved {len(all_docs)} unique documents")
    
    # 4. Confidence check
    if not all_docs:
        confidence = "NONE"
    else:
        best_score = max(score for _, score in all_docs.values())
        if best_score > 0.7:
            confidence = "HIGH"
        elif best_score > 0.4:
            confidence = "MEDIUM"
        else:
            confidence = "LOW"
    
    print(f"  Step 4 — Confidence: {confidence}")
    
    # 5. Generate response
    print(f"  Step 5 — Generate response:\n")
    
    if confidence == "NONE":
        print(f"  ┌─ RESPONSE ─────────────────────────────────────────────┐")
        print(f"  │ NOT FOUND IN DATABASE. Please consult a specialist.   │")
        print(f"  └───────────────────────────────────────────────────────┘")
        return
    
    print(f"  ┌─ RESPONSE ─────────────────────────────────────────────┐")
    print(f"  │ Retrieved documents:                                    │")
    for doc_id, (doc, score) in sorted(all_docs.items()):
        print(f"  │   [{doc_id}] {doc['title']:<40} │")
        # Show relevant excerpt
        excerpt = doc['text'][:70]
        print(f"  │     '{excerpt}...' │")
    print(f"  │                                                         │")
    print(f"  │ Confidence: {confidence:<45} │")
    if confidence == "LOW":
        print(f"  │ ⚠ LOW CONFIDENCE — recommend physician review         │")
    print(f"  └───────────────────────────────────────────────────────┘")

# Test cases
medical_rag_pipeline("What is the maximum metformin dose for eGFR 35?")
print("\n" + "─" * 60)
medical_rag_pipeline("Patient on metformin with eGFR 35 needs CT with contrast. What to do?")

---
## Summary: Medical-Grade RAG Checklist

| Technique | What It Does | Why It Matters |
|-----------|-------------|----------------|
| **Metadata filtering** | Narrow search by drug/category/date | Eliminates irrelevant docs before search |
| **Hybrid search** | Vector + keyword combined | Catches exact terms (doses, lab values) |
| **Re-ranking** | Cross-encoder rescores candidates | Much more precise than embedding alone |
| **Query decomposition** | Split complex questions | Gets ALL relevant docs, not just one |
| **Forced citation** | Every claim must cite a source | Prevents hallucination, enables audit |
| **Confidence scoring** | Know when you don't know | Prevents dangerous guesses |
| **Temporal awareness** | Prefer recent documents | Old guidelines can be wrong |

```
Basic RAG accuracy:     ~70% (fine for chatbots)
With all techniques:    ~95% (required for medicine)
Remaining 5%:           human review (non-negotiable)
```

The key insight: accuracy in RAG is NOT about a better embedding model.
It's about **engineering discipline** — filters, re-ranking, confidence checks,
citations, and knowing when to say "I don't know."